# Urethanase-Boltz2
Run the UMG-SP2/A1D5H retrospective co-folding benchmark with Boltz-2. A GPU runtime is recommended.

**Before running:** choose **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU). Boltz currently requires Python 3.10–3.12; the setup cell checks this explicitly.

In [ ]:
import sys, subprocess
print('Python:', sys.version)
if not ((3, 10) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError('Boltz requires Python >=3.10,<3.13. This Colab runtime is using an incompatible Python version.')

subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/jjimenezgar/Urethanase-Boltz2.git'], check=True)
%cd Urethanase-Boltz2

print('\nInstalling the small benchmark package...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--disable-pip-version-check'], check=True)
print('\nInstalling Boltz + CUDA dependencies (first install can take several minutes)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'boltz[cuda]', '--disable-pip-version-check', '--progress-bar', 'on'], check=True)
print('\nBoltz installation complete.')
subprocess.run(['boltz', '--help'], check=True, stdout=subprocess.DEVNULL)
print('Boltz CLI: OK')

In [ ]:
!python scripts/fetch_reference.py
!python scripts/build_boltz_input.py
!cat inputs/umg_sp2_a1d5h.yaml

## Boltz-2 inference
No experimental structure/template or binding-pocket constraint is supplied to Boltz. The X-ray structure remains evaluation-only.

In [ ]:
!boltz predict inputs/umg_sp2_a1d5h.yaml --use_msa_server --out_dir results/boltz2

In [ ]:
from pathlib import Path
predictions = sorted(Path('results/boltz2/predictions/umg_sp2_a1d5h').glob('*_model_0.cif'))
assert predictions, 'Boltz-2 prediction CIF not found; inspect the previous cell output.'
pred = str(predictions[0])
print('Evaluating:', pred)
!urethanase-boltz2 evaluate --reference data/reference/8XTC.cif --prediction $pred --ligand A1D5H --output results/metrics.json
!cat results/metrics.json